In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
import seaborn as sns

In [ ]:
df = pd.read_csv("/kaggle/input/playground-series-s5e12/train.csv")
"df loaded"

In [ ]:
df

In [ ]:
r, c = df.shape
print(r, c)

In [ ]:
# print(df.isnull().sum())
"""0 missing values"""

In [ ]:
df["diagnosed_diabetes"].value_counts()

In [ ]:
df["diagnosed_diabetes"].value_counts(normalize=True)
# gives percentage of each count on decimal format

In [ ]:
# checking corelation of attributes
correlations = df.corr(numeric_only=True)['diagnosed_diabetes'].sort_values(ascending=False)

In [ ]:
correlations

* # checking for categirical attributes for correlation

In [ ]:
df.select_dtypes(include='object')

In [ ]:
df['gender'].value_counts()

In [ ]:
df['ethnicity'].value_counts()

In [ ]:
df['smoking_status'].value_counts()

In [ ]:
categorical_cols = ['gender', 'ethnicity', 'education_level', 
                    'income_level', 'smoking_status', 'employment_status']
# using dummies
df_final = pd.get_dummies(df, columns=categorical_cols)
correlations = df_final.corr()['diagnosed_diabetes'].sort_values(ascending=False)

In [ ]:
"total cols = 44"
correlations


In [ ]:
import seaborn as sns
"imported seaborn"

In [ ]:
plt.figure(figsize = (10, 5))
sns.barplot(data=df, x='smoking_status', y='diagnosed_diabetes')
plt.title('Is Smoking really a clue?')
plt.show()

* ## looking for correlation

In [ ]:
# We only pick the top 15 most important columns so the map isn't too crowded
top_cols = correlations.head(15).index
plt.figure(figsize=(12, 8))

# Create the Heatmap
sns.heatmap(df_final[top_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('The Correlation Map (Top 15 Clues)')
plt.show()

In [ ]:
# We only pick the top 15 most important columns so the map isn't too crowded
top_cols = correlations.iloc[15:30].index
plt.figure(figsize=(12, 8))

# Create the Heatmap
sns.heatmap(df_final[top_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('The Correlation Map (16 to 30 Clues)')
plt.show()

In [ ]:
# We only pick the top 15 most important columns so the map isn't too crowded
top_cols = correlations.iloc[30:].index
plt.figure(figsize=(12, 8))

# Create the Heatmap
sns.heatmap(df_final[top_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('The Correlation Map ( 30 to 44 Clues)')
plt.show()

In [ ]:
x = df_final.drop(['diagnosed_diabetes', 'id', 'bmi', 'cholesterol_total'], axis=1)
y = df_final['diagnosed_diabetes']

x.shape

In [ ]:
from sklearn.model_selection import train_test_split  # The tool to split data
from sklearn.linear_model import LogisticRegression    # The "Student" model
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
"imported"

In [ ]:
# 2. Split the data
# 80% to study, 20% for the practice test
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
# 1. Scaling the data (makes all numbers similar in size)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)

# 2. Run the model again with the scaled data
# We keep max_iter=1000 just to be safe
model = LogisticRegression(max_iter=1000)
model.fit(x_train_scaled, y_train)

# 3. Get new predictions
val_probs = model.predict_proba(x_val_scaled)[:, 1]

# 4. Check the new score
new_score = roc_auc_score(y_val, val_probs)
print(f"New Logistic Regression Score: {new_score:.4f}")

**Now creating submission: Logistic regression model**

In [ ]:
# 1. Load the secret test data
test_df = pd.read_csv('/kaggle/input/playground-series-s5e12/test.csv')

test_dummies = pd.get_dummies(test_df)

# (Ensuring it matches the 'x' columns exactly)
x_test = test_dummies[x.columns]
# 4. Scale 
x_test_scaled = scaler.transform(x_test)
# 5. probabilities
final_probs = model.predict_proba(x_test_scaled)[:, 1]

# 6.submission file
submission = pd.DataFrame({
    'id': test_df['id'],
    'diagnosed_diabetes': final_probs
})

# 7. Save to CSV
submission.to_csv('submission.csv', index=False)
print("Your submission file is ready!")

**Using XGBoost model now.**

In [ ]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

# 1. Select all features except ID and the Answer
#  df_final : categorical to numeric 
x = df_final.drop(['diagnosed_diabetes', 'id'], axis=1)
y = df_final['diagnosed_diabetes']

# 2. Split the data (80% Study, 20% Practice Test)
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
# 3. Create the XGBoost Model
# n_estimators=500 means we will build 500 trees
# learning_rate=0.05 helps the model learn slowly and carefully
# max_depth=6 keeps the trees from getting too complicated
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,  # Uses all your Kaggle CPU cores
    random_state=42,
    eval_metric='auc'
)

# 4. Train the model
# We tell it to watch the 'validation set' to avoid over-studying (overfitting)
xgb_model.fit(
    x_train, y_train,
    eval_set=[(x_val, y_val)],
    verbose=50  # Shows us progress every 50 trees
)

In [ ]:
# 5. probabilities for the practice test
xgb_probs = xgb_model.predict_proba(x_val)[:, 1]
score = roc_auc_score(y_val, xgb_probs)
print(f"\n--- XGBoost ROC AUC Score: {score:.4f} ---")


#6. Increase figsize height significantly based on the number of features (e.g., 42)
fig, ax = plt.subplots(figsize=(10, 12)) 

# Use the 'ax' parameter to plot within our defined figure size
xgb.plot_importance(xgb_model, max_num_features=42, importance_type='weight', ax=ax)

plt.title('Top Most Important Features (XGBoost)', fontsize=15)
plt.yticks(fontsize=10) # Adjust font size of feature names
plt.tight_layout()      # Automatically adjusts subplot params for better fit
plt.show()

In [ ]:
# Create new 'Super Clues' in your original dataframe
df_final['age_bmi_risk'] = df_final['age'] * df_final['bmi']
df_final['bp_ratio'] = df_final['systolic_bp'] / (df_final['diastolic_bp'] + 1)
df_final['cholesterol_ratio'] = df_final['ldl_cholesterol'] / (df_final['hdl_cholesterol'] + 1)


# Update your X and re-split
X = df_final.drop(['diagnosed_diabetes', 'id'], axis=1)
# X = df_final.drop(['diagnosed_diabetes', 'id','ethnicity_Black', 'education_level_Highschool', 'ethnicity_White', 'employment_status_Employed', 'gender_Female'], axis=1)
y = df_final['diagnosed_diabetes']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Now re-run your 'Pro' XGBoost model!

In [ ]:
# 1. Use more trees and let the model stop itself
# We add 'early_stopping_rounds' to prevent over-studying
xgb_model_pro = xgb.XGBClassifier(
    n_estimators=5000, 
    learning_rate=0.03, # Lower learning rate is usually more accurate
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=50 # Stops if no improvement for 50 rounds
)

# 2. Re-train
xgb_model_pro.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=200 #after 100 tree show progress
)

# 3. Check new score
pro_preds = xgb_model_pro.predict_proba(X_val)[:, 1]
print(f"Pro XGBoost Score: {roc_auc_score(y_val, pro_preds):.4f}")

In [ ]:

#6. Increase figsize height significantly based on the number of features (e.g., 42)
fig, ax = plt.subplots(figsize=(10, 12)) 

# Use the 'ax' parameter to plot within our defined figure size
xgb.plot_importance(xgb_model, max_num_features=42, importance_type='weight', ax=ax)

plt.title('Top Most Important Features (XGBoost)', fontsize=15)
plt.yticks(fontsize=10) # Adjust font size of feature names
plt.tight_layout()      # Automatically adjusts subplot params for better fit
plt.show()

In [ ]:
# # checking noisy features to improve acuracy
# # 1. Get feature importance
# importances = xgb_model_pro.feature_importances_
# feature_names = X.columns

# # 2. Create a table of clues and their scores
# importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
# importance_df = importance_df.sort_values(by='importance', ascending=False)

# # 3. Identify features with VERY low importance (e.g., bottom 5)
# low_importance_features = importance_df.tail(5)['feature'].tolist()

# print("Suggested features to remove (Bottom 5):")
# print(low_importance_features)

In [ ]:
import lightgbm as lgb

# 1. Setup the model
# LightGBM is famous for being fast and accurate
lgb_model = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

# 2. Train (Make sure X_train and X_val are the 'Fixed' versions from last step)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

# 3. Score it
lgb_probs = lgb_model.predict_proba(X_val)[:, 1]
print(f"LightGBM Score: {roc_auc_score(y_val, lgb_probs):.4f}")

In [ ]:
# 1. Get the probabilities from both models on the validation set
xgb_preds = xgb_model_pro.predict_proba(X_val)[:, 1]
lgb_preds = lgb_model.predict_proba(X_val)[:, 1]

# 2. Blend them (Give a bit more weight to the stronger XGBoost model)
# 60% XGBoost + 40% LightGBM
blended_preds = (0.6 * xgb_preds) + (0.4 * lgb_preds)

# 3. Check the new team score
blended_score = roc_auc_score(y_val, blended_preds)
print(f"Blended ROC AUC Score: {blended_score:.4f}")

In [ ]:
# 1. Prepare the test data (Same features as X)
test_df = pd.read_csv('/kaggle/input/playground-series-s5e12/test.csv')

# Create the same Super Clues
test_df['age_bmi_risk'] = test_df['age'] * test_df['bmi']
test_df['bp_ratio'] = test_df['systolic_bp'] / (test_df['diastolic_bp'] + 1)
test_df['cholesterol_ratio'] = test_df['ldl_cholesterol'] / (test_df['hdl_cholesterol'] + 1)

# Match columns exactly (dummies)
test_dummies = pd.get_dummies(test_df)
X_test_final = test_dummies.reindex(columns=X.columns, fill_value=0)

# 2. Get predictions from both models
final_xgb = xgb_model_pro.predict_proba(X_test_final)[:, 1]
final_lgb = lgb_model.predict_proba(X_test_final)[:, 1]

# 3. Blend for the final submission
final_blend = (0.6 * final_xgb) + (0.4 * final_lgb)

# 4. Create CSV
submission = pd.DataFrame({'id': test_df['id'], 'diagnosed_diabetes': final_blend})
submission.to_csv('submission_blend.csv', index=False)
print("Blended submission is ready!")

**Trying other than Dummies**

In [ ]:
# 1. Define the categorical columns again
cat_features = ['gender', 'ethnicity', 'education_level', 'income_level', 'smoking_status', 'employment_status']

# 2. Use the Training set to calculate the "Risk Map" 
# (We only use the training set to avoid "cheating")
for col in cat_features:
    # Calculate average diabetes risk for each group
    risk_map = df.iloc[X_train.index].groupby(col)['diagnosed_diabetes'].mean()
    
    # Apply this map to both training and validation sets
    X_train[f'{col}_risk'] = df.iloc[X_train.index][col].map(risk_map)
    X_val[f'{col}_risk'] = df.iloc[X_val.index][col].map(risk_map)

# 3. Drop the old dummy columns (the 0s and 1s) to keep it clean
# This makes your model focus on the actual risk numbers
X_train_encoded = X_train.drop(columns=[c for c in X_train.columns if any(word in c for word in cat_features) and '_risk' not in c])
X_val_encoded = X_val.drop(columns=[c for c in X_val.columns if any(word in c for word in cat_features) and '_risk' not in c])

**creating adavanced features**

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def create_advanced_features(df):
    # 1. Logarithmic Lipid Risk (AIP) - The "Gold Standard"
    # Adding epsilon to avoid division by zero if HDL is 0 (unlikely but safe)
    df['AIP'] = np.log10(df['triglycerides'] / (df['hdl_cholesterol'] + 1))
    
    # 2. Castelli Risk Indices
    df['CRI_1'] = df['cholesterol_total'] / (df['hdl_cholesterol'] + 1)
    df['non_hdl'] = df['cholesterol_total'] - df['hdl_cholesterol']
    
    # 3. Hemodynamic Stress
    df['MAP'] = (df['systolic_bp'] + 2 * df['diastolic_bp']) / 3
    df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']
    # Interaction of Heart Rate and Pressure (Cardiovascular Workload)
    df['cardio_stress'] = df['heart_rate'] * df['systolic_bp'] / 100

    # 4. Non-Linear Sleep Risk (U-Shape Capture)
    # Captures risk of both <6h and >9h sleep
    df['sleep_deviation'] = np.abs(df['sleep_hours_per_day'] - 7.5)

    # 5. Anthropometric Risk Proxy
    # Estimating Visceral Adiposity intensity
    df['waist_bmi_interaction'] = df['waist_to_hip_ratio'] * df['bmi']
    
    # 6. Lifestyle Composite (The "Bad Habits" Multiplier)
    # (Screen Time * Alcohol) / (Diet * Activity)
    # Activity converted to hours for scale balance
    activity_in_hours = df['physical_activity_minutes_per_week'] / 60
    df['lifestyle_deficit'] = (df['screen_time_hours_per_day'] * (df['alcohol_consumption_per_week'] + 1)) / \
                              ((activity_in_hours * df['diet_score']) + 1)
                              
    return df
"done..."

In [ ]:
# 1. Load Data
df = pd.read_csv("/kaggle/input/playground-series-s5e12/train.csv")

# 2. Add Super Clues
df = create_advanced_features(df)
# df = pd.concat([df, super_clues], axis=1)


from sklearn.model_selection import KFold
# -------------------------
# 3. SAFE K-FOLD TARGET ENCODING (NO ERRORS)
# -------------------------
def target_encode_kfold(df, col, target, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    global_mean = df[target].mean()

    # Force 1D column
    col_series = df[col].astype(str)

    encoded = pd.Series(index=df.index, dtype=float)

    for tr, val in kf.split(df):
        train_col = col_series.iloc[tr]
        train_target = df[target].iloc[tr]

        means = train_target.groupby(train_col).mean()
        encoded.iloc[val] = col_series.iloc[val].map(means)

    return encoded.fillna(global_mean)

cat_cols = [
    'gender', 'ethnicity', 'education_level',
    'income_level', 'smoking_status', 'employment_status'
]

for col in cat_cols:
    df[f'{col}_risk'] = target_encode_kfold(df, col, 'diagnosed_diabetes')

# Drop original categorical columns
df.drop(columns=cat_cols, inplace=True)

# -------------------------
# 4. Prepare X and y
# -------------------------
X = df.drop(columns=['diagnosed_diabetes', 'id'])
y = df['diagnosed_diabetes']

# -------------------------
# 5. Stratified K-Fold + LightGBM
# -------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lgb_scores = []

for tr, val in skf.split(X, y):
    X_tr, X_val = X.iloc[tr], X.iloc[val]
    y_tr, y_val = y.iloc[tr], y.iloc[val]

    lgb_model = lgb.LGBMClassifier(
        n_estimators=4000,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(50)]
    )

    preds = lgb_model.predict_proba(X_val)[:, 1]
    lgb_scores.append(roc_auc_score(y_val, preds))

print("LightGBM Mean AUC:", np.mean(lgb_scores))

# --------------------

In [ ]:
# k fold xgb
xgb_scores = []

for tr, val in skf.split(X, y):
    X_tr, X_val = X.iloc[tr], X.iloc[val]
    y_tr, y_val = y.iloc[tr], y.iloc[val]

    xgb_model = xgb.XGBClassifier(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.5,
        reg_lambda=1.0,
        eval_metric='auc',
        early_stopping_rounds=50,
        random_state=42
    )

    xgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds = xgb_model.predict_proba(X_val)[:, 1]
    xgb_scores.append(roc_auc_score(y_val, preds))

print("XGBoost Mean AUC:", np.mean(xgb_scores))


In [ ]:
import joblib
joblib.dump(xgb_model, "xgb_model")
joblib.dump(lgb_model, "lgb_model")

print(joblib.load("xgb_model"))
print(joblib.load("lgb_model"))